# One scientific operation per pipeline module

Each section starts with one visible input and calls the module that performs the scientific operation. Experiment resolution, pipeline runners, and saved-run dispatch are intentionally absent. The baseline YAML is read directly so that every selected method resource remains visible.

In [ ]:
import json
import tempfile
from pathlib import Path

import numpy as np

from grammar_kt import canonical, items, kc, kt, normalisation, qmatrix, realisation, simulation, source
from grammar_kt.io import ROOT, read_json, read_jsonl, read_yaml
from grammar_kt.records import kc_opportunity, observable_interaction

settings = read_yaml(ROOT / "experiments" / "base.yaml")

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))

## Source

Project one source descriptor onto the five fields visible to Phase 1.

In [ ]:
source_descriptor = read_jsonl(ROOT / "modules/source/fixtures/core.jsonl")[0]
phase1_input = source.phase1_record(source_descriptor)
show({"source_descriptor": source_descriptor, "phase1_input": phase1_input})

## Normalisation

Annotate one fixed descriptor with the selected Phase-1/Phase-2 prompts and backend. The direct call renders the prompts, invokes the model, validates both phases, and retains the evidence.

In [ ]:
mapping_input = next(
    row for row in read_jsonl(ROOT / "modules/normalisation/fixtures/core.jsonl")
    if row["fixture_label"] == "passive"
)
method = settings["normalisation"]
evidence_dir = Path(tempfile.mkdtemp(prefix="grammar-kt-notebook-normalisation-"))
mapping_result = normalisation.normalise_one(
    mapping_input,
    phase1_template=(ROOT / method["phase1_prompt"]).read_text(encoding="utf-8"),
    phase2_template=(ROOT / method["phase2_prompt"]).read_text(encoding="utf-8"),
    backend_settings=read_yaml(ROOT / method["backend"]),
    max_attempts=method["max_attempts"],
    output=evidence_dir,
)
show({
    "source_descriptor": mapping_input,
    "mapping": mapping_result["output"],
    "phase2_routing_reason": mapping_result["phase2_routing_reason"],
    "evidence_directory": mapping_result["evidence_directory"],
})

## Canonical cells

Complete mappings become exact GrammarCells; identical cells are deduplicated while their source edges are retained.

In [ ]:
complete_mapping = {
    "egp_id": "FIX_CANONICAL",
    "result": "complete",
    "cells": [{"tense": "past", "aspect": "none", "voice": "passive",
               "polarity": "positive", "clause": "declarative", "modal": "none"}],
    "note": None,
}
canonical_cells, source_edges = canonical.build([complete_mapping])
show({"complete_mapping": complete_mapping, "canonical_cells": canonical_cells, "source_edges": source_edges})

## Realisation

Realise and validate one GrammarCell/RealizationSpec pair through the morphology, auxiliary-chain, and clause operations.

In [ ]:
realisation_input = next(
    row for row in read_jsonl(ROOT / "modules/realisation/fixtures/core.jsonl")
    if row["fixture_label"] == "lexical_present_question"
)
realisation_result = realisation.run_one(realisation_input)
show({"cell_and_spec": realisation_input, "derivation": realisation_result["output"],
      "valid": realisation_result["valid"], "errors": realisation_result["errors"]})

## KC selection

Load the declared policy and evaluate its activation rules against one grammatical opportunity. The returned explanation contains literal rule evidence.

In [ ]:
opportunity = kc_opportunity(read_json(ROOT / "modules/kc/fixtures/perfect_progressive.json"))
policy = kc.load_policy(ROOT / settings["kc"]["policy"])
activation = kc.apply_policy(policy, opportunity)
projections, kc_inventory = kc.materialize_inventory(policy, [opportunity])
show({"opportunity": opportunity, "activation": activation, "kc_inventory": kc_inventory})

## Items

Check one controlled-transformation item against the same deterministic realiser used during generation.

In [ ]:
item_input = next(
    row for row in read_jsonl(ROOT / "modules/items/fixtures/core.jsonl")
    if row["fixture_label"] == "valid_deterministic_item"
)
item_result = items.evaluate_fixture(item_input)
show({"item": item_input, "realised_answer": item_result["output"]["surface"],
      "valid": item_result["valid"], "errors": item_result["errors"]})

## Q-matrix

Project accepted item labels from the frozen cell/KC assignment into a matrix, item–KC edges, and diagnostics.

In [ ]:
q_item = {"item_id": "ITEM_DEMO", "canonical_cell_id": "CELL_DEMO",
          "all_kc_ids": ["KC_FINITE_PRESENT"],
          "realization_spec": {"realization_id": "REAL_DEMO"},
          "source_descriptor_ids": ["FIX_Q"]}
q_card = {"kc_id": "KC_FINITE_PRESENT",
          "activation_rule": {"cell": {"tense": "present"}}}
q_projection = {"canonical_cell_id": "CELL_DEMO", "kc_ids": ["KC_FINITE_PRESENT"]}
q_columns, q_rows, q_edges, q_audit = qmatrix.build([q_item], [q_card], [q_projection])
show({"item": q_item, "projection": q_projection,
      "matrix": {"columns": q_columns, "rows": q_rows}, "edges": q_edges, "audit": q_audit})

## Simulation

Load the tiny item/Q fixtures, derive chronological split boundaries, and run the unchanged response/mastery equations for one learner.

In [ ]:
simulation_dir = ROOT / "modules/simulation"
simulation_items = read_jsonl(simulation_dir / "fixtures/accepted_items.jsonl")
simulation_kcs, q_by_item = simulation.read_q_matrix(simulation_dir / "fixtures/q_matrix.csv")
parameters = read_json(ROOT / settings["simulation"]["parameters"])
parameters["seed"] = settings["simulation"]["seed"]
event_count = len(simulation_items) * parameters["item_passes_per_learner"]
train_end, validation_end = simulation.split_boundaries(
    event_count, parameters["train_fraction"], parameters["validation_fraction"]
)
observed, oracle, learners, learner_parameters = simulation.simulate_records(
    parameters,
    {row["item_id"]: row for row in simulation_items},
    q_by_item,
    simulation_kcs,
    train_end,
    validation_end,
    target_learner="L0001",
)
show({"learner": learners[0], "observable_interactions": observed,
      "oracle_rows_retained_separately": len(oracle)})

## Knowledge tracing

Validate one observable history, build pre-event features, compute the empirical and BKT baselines, and evaluate only validation/test predictions.

In [ ]:
outcomes = [0, 1, 0, 1, 1, 0]
splits = ["train", "train", "validation", "validation", "test", "test"]
interactions = [
    {"event_id": f"EVENT_DEMO_{index:03d}", "learner_id": "L_DEMO",
     "item_id": "ITEM_DEMO", "sequence_index": index,
     "timestamp": f"2026-01-01T00:{index:02d}:00+00:00", "correct": outcome,
     "kc_ids": ["KC_FINITE_PRESENT"],
     "opportunity_indices": {"KC_FINITE_PRESENT": index},
     "canonical_cell_id": "CELL_DEMO", "item_difficulty": 0.1,
     "dataset_split": split}
    for index, (outcome, split) in enumerate(zip(outcomes, splits), 1)
]
for row in interactions:
    observable_interaction(row, label=row["event_id"])

kc_ids = ["KC_FINITE_PRESENT"]
kt_parameters = read_json(ROOT / settings["kt"]["parameters"])
alpha = kt_parameters["empirical"]["alpha"]
beta = kt_parameters["empirical"]["beta"]
features, targets, empirical = kt.pre_event_features(interactions, kc_ids, alpha, beta)
bkt_settings = kt_parameters["bkt"]
bkt = kt.bkt_predictions(
    interactions, kc_ids, learn=bkt_settings["learn"], guess=bkt_settings["guess"],
    slip=bkt_settings["slip"], alpha=alpha, beta=beta,
)
split = np.asarray(splits)
metrics = {
    name: {part: kt.prediction_metrics(targets[split == part], predictions[split == part])
           for part in ("validation", "test")}
    for name, predictions in {"empirical": empirical, "bkt": bkt}.items()
}
show({"interactions": interactions, "feature_shape": features.shape, "metrics": metrics})